In [1]:
import pandas as pd
import datetime as dt

In [2]:
raven = pd.concat(pd.read_excel('raven.xlsx', sheet_name=None), ignore_index=True)

new_cols_stripped = []
for col in raven.columns:
    new_cols_stripped.append(col.rstrip().replace(" ","_").lower().replace("(","").replace(")",""))
raven.columns = new_cols_stripped
raven['duration'] = raven["end_s"] - raven["start_s"]
raven['date'] = raven['wav_file'].astype('str')
raven['date'] = pd.to_datetime(raven['date'], format="%y%m%d%H%M%S")

FileNotFoundError: [Errno 2] No such file or directory: 'raven.xlsx'

In [250]:
def generate_raven_spectrogram_manifest(df, spectrogram_length_s, first_two_seconds_only, high_freq_hz_max):

    detections_df = pd.DataFrame({'UTC' : pd.Series(dtype='datetime64[ns]'), 
                                  'Species' : pd.Series(dtype='str'),
                                  'Detection_TimeStamp' : pd.Series(dtype='datetime64[ns]'), 
                                  'Date' : pd.Series(dtype='datetime64[ns]'), 
                                  'audio_filename' : pd.Series(dtype='str'),
                                  'source' : pd.Series(dtype='str')})
    
    #filter detections of high freq calls
    df = df[(df['high_freq_hz'] <= 1000)]
    for index, row in df.iterrows():
        
        #get the data per value 
        
        detection_date = row['date']
        detection_time = row['time'] #date and time of the detection in the real world
        detection_wavefile = row['wav_file']
        detection_start_s = pd.to_datetime(detection_date.strftime("%Y%m%d") + row['raven_start_time'].strftime("%H%M%S"))
        detection_end_s = pd.to_datetime(detection_date.strftime("%Y%m%d") + row['raven_end_time'].strftime("%H%M%S"))
                              
   
        #generate 2 second interval 
        initial_time = detection_start_s
        intermediate_time = initial_time
           
        if first_two_seconds_only == False: #that is, we want to use the entire boxed detection
            #while the end
            
            while detection_end_s > intermediate_time:
                spectrogram_start = intermediate_time
                
                #advance it by the specified time interval
                spectrogram_end = intermediate_time + pd.Timedelta(spectrogram_length_s,'s')
                
                #propogate values
                intermediate_time = spectrogram_end

                #spectrogram_start = detection_date + pd.to_timedelta(spectrogram_start,  unit = "s")


                #print(detection_date, type(detection_date))
                spectrogram_dict = {}
                #print(str(spectrogram_start))
                spectrogram_dict['UTC'] = str(spectrogram_start) #fine
                spectrogram_dict['Species'] = 'B'
                spectrogram_dict['Detection_TimeStamp'] = spectrogram_start.strftime("%Y%m%d%H%M%S")
                spectrogram_dict['Date'] = detection_date.strftime("%Y%m%d")
                spectrogram_dict['audio_filename'] = detection_wavefile
                spectrogram_dict['source'] = "-".join(["raven_full_box",str(high_freq_hz_max)])


                detections_df = detections_df.append(spectrogram_dict, ignore_index=True)
        else:
            
            spectrogram_start = detection_start_s

            spectrogram_dict = {}            
            spectrogram_dict['UTC'] = str(spectrogram_start) #fine
            spectrogram_dict['Species'] = 'B'
            spectrogram_dict['Detection_TimeStamp'] = spectrogram_start.strftime("%Y%m%d%H%M%S")
            spectrogram_dict['Date'] = detection_date.strftime("%Y%m%d")
            spectrogram_dict['audio_filename'] = detection_wavefile
            spectrogram_dict['source'] = "-".join(["raven_first_2",str(high_freq_hz_max)])
            
            detections_df = detections_df.append(spectrogram_dict, ignore_index=True)
            
            
    return detections_df,df
           

test = generate_raven_spectrogram_manifest(raven, 2, True, 1000)
test[0]



,UTC,Species,Detection_TimeStamp,Date,audio_filename,source
0,2018-08-21 10:33:31,B,20180821103331,20180821,180821103002,raven_first_2-1000
1,2018-08-21 11:47:49,B,20180821114749,20180821,180821114502,raven_first_2-1000
2,2018-08-21 11:47:51,B,20180821114751,20180821,180821114502,raven_first_2-1000
3,2018-08-21 11:47:53,B,20180821114753,20180821,180821114502,raven_first_2-1000
4,2018-08-28 02:16:25,B,20180828021625,20180828,180828021502,raven_first_2-1000
5,2018-08-28 02:31:24,B,20180828023124,20180828,180828023002,raven_first_2-1000
6,2018-08-28 02:46:00,B,20180828024600,20180828,180828024502,raven_first_2-1000
7,2018-08-29 15:17:44,B,20180829151744,20180829,180829151502,raven_first_2-1000
8,2018-08-29 15:17:56,B,20180829151756,20180829,180829151502,raven_first_2-1000
9,2018-08-29 15:30:10,B,20180829153010,20180829,180829153002,raven_first_2-1000


Timestamp('2018-08-16 17:19:55')

TypeError: 'NoneType' object is not subscriptable

'20180816053444'